# **RESULT AFTER PREPROCESSING**


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
from functools import reduce
from scipy.stats import gaussian_kde, pearsonr
from matplotlib import gridspec
from matplotlib.lines import Line2D
from scipy.interpolate import interp1d
from PyPDF2 import PdfReader, PdfWriter
import pickle

In [2]:
EDA_FREQ = 15 # Updated frequency
BVP_FREQ = 64

new_sampling_rates = {
        "emotibit": {"Gsr": EDA_FREQ, "Bvp": BVP_FREQ, "Pr": BVP_FREQ, "Pi": BVP_FREQ},
        "shimmer": {"Gsr": EDA_FREQ, "Bvp": BVP_FREQ},
        "empatica": {"Gsr": EDA_FREQ, "Bvp": BVP_FREQ}
    }

def extract_freq(signal, device):
    device_lower = device.lower()
    if "emotibit" in device_lower and signal in new_sampling_rates["emotibit"]:
        return new_sampling_rates["emotibit"][signal]
    elif "shimmer" in device_lower and signal in new_sampling_rates["shimmer"]:
        return new_sampling_rates["shimmer"][signal]
    elif "empatica" in device_lower and signal in new_sampling_rates["empatica"]:
        return new_sampling_rates["empatica"][signal]
    else:
        return None

In [3]:
def load_events(user, phase="first"):
    try:
        events = pd.read_csv(
            f"data/{phase}/Stamps/{user}.txt", header=None, names=['Event', 'Timestamp']
        )
    except FileNotFoundError:
        return []

    return events

## Scatter and density plot

In [ ]:
def plot_scatter_pearson(data, phase, signal,
                         device_names,
                         comparisons,
                         comparisons_tick,
                         colors,
                         output_pdf='output.pdf',
                         task=None):

    try:
        users_list = list(data[device_names[0]].keys())
    except Exception as e:
        print(f"⚠️ Could not get users: {e}")
        return

    with PdfPages(output_pdf) as pdf:
        for user_id in [users_list[0]]:
            try:
                signals_ref = [data[d][user_id][signal] for d in device_names]
            except Exception as e:
                print(f"⚠️ Error loading signals for user {user_id}: {e}")
                continue

            def plot_user(devices_ref, techniques=["original"]):
                try:
                    if "df" in devices_ref[0]:
                        return
                except Exception:
                    pass

                for technique in devices_ref[0]:
                    try:
                        if technique == "original":
                            fig = plt.figure(figsize=(7, 4), dpi=200) if phase == "first" else plt.figure(
                                figsize=(8, 4), dpi=200)
                            gs = gridspec.GridSpec(1, 2, width_ratios=[4, 1], wspace=0.05) if phase == "first" else gridspec.GridSpec(
                                1, 2, width_ratios=[5, 1], wspace=0.05)

                            ax = fig.add_subplot(gs[0])
                            ax_density = fig.add_subplot(gs[1], sharey=ax)
                            ax_density.yaxis.set_visible(False)
                            ax_density.set_xticks([])
                            ax_density.set_frame_on(False)

                            user_results = {}
                            for user in users_list:
                                user_results[user] = []
                                for idx, (dev1, dev2) in enumerate(comparisons):
                                    try:
                                        df1 = reduce(lambda d, k: d[k], techniques[1:], data[dev1][user][signal])[
                                            "original"]["df"]
                                        df2 = reduce(lambda d, k: d[k], techniques[1:], data[dev2][user][signal])[
                                            "original"]["df"]

                                        events = load_events(user, phase)
                                        if task:
                                            t0 = events[events["Event"] ==
                                                        task[0]]["Timestamp"].values[0]
                                            t1 = events[events["Event"] ==
                                                        task[1]]["Timestamp"].values[0]
                                            df1 = df1[(df1['Timestamp'] >= t0) & (
                                                df1['Timestamp'] <= t1)]
                                            df2 = df2[(df2['Timestamp'] >= t0) & (
                                                df2['Timestamp'] <= t1)]

                                        if df1.empty or df2.empty:
                                            user_results[user].append(
                                                (np.nan, np.nan))
                                            continue
                                        if 'Timestamp' not in df1.columns or 'Timestamp' not in df2.columns:
                                            user_results[user].append(
                                                (np.nan, np.nan))
                                            continue

                                        min_time = max(
                                            df1['Timestamp'].min(), df2['Timestamp'].min())
                                        max_time = min(
                                            df1['Timestamp'].max(), df2['Timestamp'].max())
                                        if min_time >= max_time:
                                            user_results[user].append(
                                                (np.nan, np.nan))
                                            continue

                                        fs = extract_freq(signal, dev1)
                                        if fs is not None:
                                            num_points = int(
                                                (max_time - min_time) * fs)
                                            if num_points < 2:
                                                user_results[user].append(
                                                    (np.nan, np.nan))
                                                continue
                                        else:
                                            len1 = ((df1['Timestamp'] <= max_time) & (
                                                df1['Timestamp'] >= min_time)).sum()
                                            len2 = ((df2['Timestamp'] <= max_time) & (
                                                df2['Timestamp'] >= min_time)).sum()
                                            num_points = min(len1, len2)
                                            if num_points < 2:
                                                user_results[user].append(
                                                    (np.nan, np.nan))
                                                continue

                                        common_time = np.linspace(
                                            min_time, max_time, num_points)
                                        f1 = interp1d(df1['Timestamp'], df1[df1.columns[1]], kind='nearest',
                                                      bounds_error=False, fill_value="extrapolate")
                                        f2 = interp1d(df2['Timestamp'], df2[df2.columns[1]], kind='nearest',
                                                      bounds_error=False, fill_value="extrapolate")
                                        aligned1 = f1(common_time)
                                        aligned2 = f2(common_time)

                                        if np.all(np.isnan(aligned1)) or np.all(np.isnan(aligned2)):
                                            user_results[user].append(
                                                (np.nan, np.nan))
                                            continue

                                        r, p = pearsonr(aligned1, aligned2)
                                        user_results[user].append((r, p))
                                    except Exception as e:
                                        print(
                                            f"⚠️ Error processing {dev1}-{dev2} for user {user}: {e}")
                                        user_results[user].append(
                                            (np.nan, np.nan))
                                        continue

                            for user, values in user_results.items():
                                x_vals, y_vals = [], []
                                for idx, (r, p) in enumerate(values):
                                    if np.isnan(r):
                                        continue
                                    jitter = (np.random.rand() - 0.5) * 0.08
                                    x_pos = idx + jitter
                                    x_vals.append(x_pos)
                                    y_vals.append(r)
                                    marker = 'o' if p <= 0.05 else 'X'
                                    ax.scatter(x_pos, r, s=60, color=colors[idx % len(colors)],
                                               edgecolors='k', alpha=1 if p <= 0.05 else 1, marker=marker, zorder=2)
                                if len(x_vals) > 1:
                                    ax.plot(x_vals, y_vals, color='gray',
                                            alpha=0.7, linewidth=1, zorder=1)

                            try:
                                y_vals = np.linspace(-1, 1, 500)
                                for idx, (dev1, dev2) in enumerate(comparisons):
                                    corr_vals = [user_results[u][idx][0] for u in users_list
                                                 if not np.isnan(user_results[u][idx][0])]
                                    if corr_vals:
                                        kde = gaussian_kde(corr_vals)
                                        density = kde(y_vals)
                                        density = density / density.max() * 0.3
                                        ax_density.fill_betweenx(
                                            y_vals, 0, density, color=colors[idx % len(colors)], alpha=0.3)
                            except Exception as e:
                                print(f"⚠️ Error generating density: {e}")

                            # --- Legend and formatting ---
                            legend_elements = [
                                Line2D([0], [0], marker='o', color='k',
                                       label='p ≤ 0.05', markersize=8, linestyle=''),
                                Line2D([0], [0], marker='X', color='k',
                                       label='p > 0.05', markersize=8, linestyle='')
                            ]
                            ax.legend(handles=legend_elements,
                                      loc='lower right', fontsize=10)
                            ax.set_xticks(range(len(comparisons)))
                            ax.set_xticklabels(comparisons_tick, fontsize=12)
                            ax.set_ylim(-1, 1)
                            ax.set_ylabel(
                                "Pearson Correlation (R)", fontsize=12)
                            ax.spines['top'].set_visible(False)
                            ax.spines['right'].set_visible(False)
                            ax.grid(False)
                            pdf.savefig(fig, dpi=200, bbox_inches='tight')
                            plt.close()

                            # --- Summary page ---
                            fig_summary = plt.figure(figsize=(8, 6), dpi=200)
                            text_lines = [
                                f"Correlation Summary - Technique: {' -> '.join(techniques)}\n"
                            ]

                            for idx, (dev1, dev2) in enumerate(comparisons):
                                vals = [user_results[u][idx][0] for u in users_list if not np.isnan(
                                    user_results[u][idx][0])]
                                pvals = [user_results[u][idx][1] for u in users_list if not np.isnan(
                                    user_results[u][idx][1])]

                                if not vals:
                                    continue

                                mean_r, std_r = np.mean(vals), np.std(vals)
                                min_r, max_r = np.min(vals), np.max(vals)
                                mean_p, min_p, max_p = np.mean(
                                    pvals), np.min(pvals), np.max(pvals)

                                # Count non-significant p-values
                                nonsig_count = sum(p > 0.05 for p in pvals)

                                text_lines.append(
                                    f"{dev1.capitalize()} vs {dev2.capitalize()}:\n"
                                    f"  Mean R: {mean_r:.3f}\n"
                                    f"  Std R: {std_r:.3f}\n"
                                    f"  Min R: {min_r:.3f}, Max R: {max_r:.3f}\n"
                                    f"  Mean p-value: {mean_p:.3e}\n"
                                    f"  Min p-value: {min_p:.3e}, Max p-value: {max_p:.3e}\n"
                                    f"  Non-significant count (p > 0.05): {nonsig_count}\n"
                                )

                            fig_summary.text(0.1, 0.5, "\n".join(
                                text_lines), fontsize=12, va='center')
                            pdf.savefig(fig_summary, dpi=200,
                                        bbox_inches='tight')
                            plt.close()

                        else:
                            plot_user(
                                [ref[technique] for ref in devices_ref], techniques + [technique])

                    except Exception as e:
                        print(f"⚠️ Error in technique {technique}: {e}")
                        continue

            try:
                plot_user(signals_ref)
            except Exception as e:
                print(f"⚠️ Error processing user {user_id}: {e}")
                continue

    print(f"✅ PDF generated: {output_pdf}")

## Behaviour plot line

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
from functools import reduce
from matplotlib import gridspec


def plot_n_means(data,
                 phase,
                 device_names,
                 tasks,
                 signal='Gsr',
                 output_pdf='output.pdf',
                 n_means=8):

    try:
        users_list = list(data[device_names[0]].keys())

        # As described in the paper, some users were not considered only in GSR first phase due to signal saturation
        if signal == "Gsr" and phase == "first":
            users_list = [u for u in users_list if u not in ["13", "15", "16"]]

    except Exception as e:
        print(f"⚠️ Could not get users: {e}")
        return

    with PdfPages(output_pdf) as pdf:

        for user_id in [users_list[0]]:
            try:
                signals_ref = [data[d][user_id][signal] for d in device_names]
            except Exception as e:
                print(f"⚠️ Error loading signals for user {user_id}: {e}")
                continue

            def plot_user(devices_ref, techniques=["original"]):
                try:
                    if "df" in devices_ref[0]:
                        return
                except Exception:
                    pass

                for technique in devices_ref[0]:
                    try:
                        if technique == "original":
                            print(techniques)

                            fig = plt.figure(figsize=(11, 4), dpi=200)
                            gs = gridspec.GridSpec(
                                1, len(device_names), wspace=0.35)

                            global_y_values = []

                            for device in device_names:
                                for user in users_list:
                                    try:
                                        events = load_events(user, phase)
                                        df = reduce(lambda d, k: d[k], techniques[1:], data[device][user][signal])[
                                            "original"]["df"]

                                        for start_ev, end_ev, _ in tasks:
                                            t_start = events[events["Event"] ==
                                                             start_ev]["Timestamp"].values[0]
                                            t_end = events[events["Event"] ==
                                                           end_ev]["Timestamp"].values[0]
                                            segment_length = (
                                                t_end - t_start) / n_means
                                            for i in range(n_means):
                                                seg_start = t_start + i * segment_length
                                                seg_end = t_start + \
                                                    (i+1) * segment_length
                                                df_seg = df[(df["Timestamp"] >= seg_start) & (
                                                    df["Timestamp"] < seg_end)]
                                                if df_seg.empty:
                                                    global_y_values.append(
                                                        np.nan)
                                                else:
                                                    global_y_values.append(
                                                        np.log(df_seg[df_seg.columns[1]].mean() + 1))
                                    except Exception:
                                        continue

                            if len(global_y_values) > 0:
                                global_y_min = np.nanmin(
                                    global_y_values) - 0.05
                                global_y_max = np.nanmax(
                                    global_y_values) + 0.05
                            else:
                                global_y_min, global_y_max = 0, 1

                            for idx, device in enumerate(device_names):
                                try:
                                    ax = fig.add_subplot(gs[idx])
                                    ax.set_ylim(global_y_min, global_y_max)

                                    if idx == 0:
                                        ax.set_ylabel(
                                            "log(mean + 1 (µs))" if signal == "Gsr" else "log(mean + 1 (bpm))")
                                    else:
                                        ax.set_ylabel("")

                                    # Tick positions: center of each phase
                                    tick_positions = [
                                        i * n_means + n_means/2 - 0.5 for i in range(len(tasks))]
                                    ax.set_xticks(tick_positions)
                                    ax.set_xticklabels(
                                        [p[2] for p in tasks], rotation=28, ha='right')

                                    # Vertical lines between phases (except the first)
                                    for i in range(1, len(tasks)):
                                        ax.axvline(
                                            i * n_means - 0.5, color='black', alpha=0.2, linestyle='--', linewidth=1)

                                    # ax.grid(True, alpha=0.3)

                                    # Gray lines for each user (without points)
                                    for user in users_list:
                                        try:
                                            events = load_events(user, phase)
                                            df = reduce(lambda d, k: d[k], techniques[1:], data[device][user][signal])[
                                                "original"]["df"]
                                            y_all = []
                                            for start_ev, end_ev, _ in tasks:
                                                t_start = events[events["Event"] ==
                                                                 start_ev]["Timestamp"].values[0]
                                                t_end = events[events["Event"] ==
                                                               end_ev]["Timestamp"].values[0]
                                                segment_length = (
                                                    t_end - t_start) / n_means
                                                for i in range(n_means):
                                                    seg_start = t_start + i * segment_length
                                                    seg_end = t_start + \
                                                        (i+1) * segment_length
                                                    df_seg = df[(df["Timestamp"] >= seg_start) & (
                                                        df["Timestamp"] < seg_end)]
                                                    if df_seg.empty:
                                                        y_all.append(np.nan)
                                                    else:
                                                        y_all.append(
                                                            np.log(df_seg[df_seg.columns[1]].mean() + 1))
                                            ax.plot(
                                                np.arange(len(tasks)*n_means), y_all, color='gray', linewidth=0.8, alpha=0.5)
                                        except Exception:
                                            continue

                                    # Red mean line across all users
                                    all_users_means = []
                                    for user in users_list:
                                        try:
                                            events = load_events(user, phase)
                                            df = reduce(lambda d, k: d[k], techniques[1:], data[device][user][signal])[
                                                "original"]["df"]
                                            y_user = []
                                            for start_ev, end_ev, _ in tasks:
                                                t_start = events[events["Event"] ==
                                                                 start_ev]["Timestamp"].values[0]
                                                t_end = events[events["Event"] ==
                                                               end_ev]["Timestamp"].values[0]
                                                segment_length = (
                                                    t_end - t_start) / n_means
                                                for i in range(n_means):
                                                    seg_start = t_start + i * segment_length
                                                    seg_end = t_start + \
                                                        (i+1) * segment_length
                                                    df_seg = df[(df["Timestamp"] >= seg_start) & (
                                                        df["Timestamp"] < seg_end)]
                                                    if df_seg.empty:
                                                        y_user.append(np.nan)
                                                    else:
                                                        y_user.append(
                                                            np.log(df_seg[df_seg.columns[1]].mean() + 1))
                                            all_users_means.append(y_user)
                                        except Exception:
                                            continue

                                    if len(all_users_means) > 0:
                                        mean_red = np.nanmean(
                                            np.array(all_users_means, dtype=float), axis=0)
                                        ax.plot(np.arange(
                                            len(tasks)*n_means), mean_red, marker=None, color='red', linewidth=2)

                                except Exception as e:
                                    print(
                                        f"⚠️ Error generating plot for {device}: {e}")
                                    continue

                            pdf.savefig(fig, dpi=200, bbox_inches='tight')
                            plt.close()

                        else:
                            try:
                                plot_user([ref[technique] for ref in devices_ref],
                                          techniques=techniques + [technique])
                            except Exception as e:
                                print(
                                    f"⚠️ Recursive error in technique {technique}: {e}")
                                continue

                    except Exception as e:
                        print(
                            f"⚠️ General error in technique {technique}: {e}")
                        continue

            try:
                plot_user(signals_ref)
            except Exception as e:
                print(f"⚠️ Error processing user {user_id}: {e}")
                continue

    print(f'✅ PDF generated: {output_pdf}')

## Tabla

In [ ]:
import numpy as np
from functools import reduce
from scipy.stats import pearsonr
from scipy.interpolate import interp1d


def get_table_data(data, phase, signal,
                   device_names, comparisons, comparisons_name, crop=None):

    tecniques_comparisons = {}

    try:
        users_list = list(data[device_names[0]].keys())
    except Exception as e:
        print(f"⚠️ Could not get users: {e}")
        return

    for user_id in [users_list[0]]:
        try:
            signals_ref = [data[d][user_id][signal] for d in device_names]
        except Exception as e:
            print(f"⚠️ Error loading signals for user {user_id}: {e}")
            continue

        def plot_user(devices_ref, techniques=["original"]):
            try:
                if "df" in devices_ref[0]:
                    return
            except Exception:
                pass
            for technique in devices_ref[0]:
                try:
                    if technique == "original":

                        user_results = {}
                        for user in users_list:
                            user_results[user] = []
                            for idx, (dev1, dev2) in enumerate(comparisons):
                                try:
                                    df1 = reduce(lambda d, k: d[k], techniques[1:], data[dev1][user][signal])[
                                        "original"]["df"]
                                    df2 = reduce(lambda d, k: d[k], techniques[1:], data[dev2][user][signal])[
                                        "original"]["df"]

                                    events = load_events(user, phase)
                                    if crop:
                                        t0 = events[events["Event"] ==
                                                    crop[0]]["Timestamp"].values[0]
                                        t1 = events[events["Event"] ==
                                                    crop[1]]["Timestamp"].values[0]
                                        df1 = df1[(df1['Timestamp'] >= t0) & (
                                            df1['Timestamp'] <= t1)]
                                        df2 = df2[(df2['Timestamp'] >= t0) & (
                                            df2['Timestamp'] <= t1)]

                                    if df1.empty or df2.empty:
                                        continue
                                    if 'Timestamp' not in df1.columns or 'Timestamp' not in df2.columns:
                                        continue

                                    min_time = max(
                                        df1['Timestamp'].min(), df2['Timestamp'].min())
                                    max_time = min(
                                        df1['Timestamp'].max(), df2['Timestamp'].max())
                                    if min_time >= max_time:
                                        continue

                                    fs = extract_freq(signal, dev1)
                                    if fs is not None:
                                        num_points = int(
                                            (max_time - min_time) * fs)
                                        if num_points < 2:
                                            continue
                                    else:
                                        len1 = ((df1['Timestamp'] <= max_time) & (
                                            df1['Timestamp'] >= min_time)).sum()
                                        len2 = ((df2['Timestamp'] <= max_time) & (
                                            df2['Timestamp'] >= min_time)).sum()
                                        num_points = min(len1, len2)
                                        if num_points < 2:
                                            continue

                                    common_time = np.linspace(
                                        min_time, max_time, num_points)
                                    f1 = interp1d(df1['Timestamp'], df1[df1.columns[1]], kind='nearest',
                                                  bounds_error=False, fill_value="extrapolate")
                                    f2 = interp1d(df2['Timestamp'], df2[df2.columns[1]], kind='nearest',
                                                  bounds_error=False, fill_value="extrapolate")

                                    aligned1 = f1(common_time)
                                    aligned2 = f2(common_time)
                                    if np.all(np.isnan(aligned1)) or np.all(np.isnan(aligned2)):
                                        continue

                                    r, p = pearsonr(aligned1, aligned2)
                                    user_results[user].append((r, p))
                                except Exception as e:
                                    print(
                                        f"⚠️ Error processing {dev1}-{dev2} for user {user}: {e}")
                                    user_results[user].append(
                                        (np.nan, np.nan))
                                    continue

                        tecniques_comparisons[" -> ".join(techniques)] = {}
                        for idx, (dev1, dev2) in enumerate(comparisons):
                            vals = [user_results[u][idx][0] for u in users_list if not np.isnan(
                                user_results[u][idx][0])]
                            pvals = [user_results[u][idx][1] for u in users_list if not np.isnan(
                                user_results[u][idx][1])]

                            if not vals:
                                continue
                            mean_r = np.mean(vals)
                            non_significant = np.sum(np.array(pvals, ) > 0.05)

                            tecniques_comparisons[" -> ".join(techniques)][f"{comparisons_name[dev1]} vs {comparisons_name[dev2]}"] = {
                                "mean_r": mean_r,
                                "non_significant": non_significant,
                            }

                    else:
                        plot_user(
                            [ref[technique] for ref in devices_ref], techniques + [technique])
                except Exception as e:
                    print(
                        f"⚠️ Error in technique {user_id} {signal} {technique}: {e}")
                    continue
        try:
            plot_user(signals_ref)
        except Exception as e:
            print(f"⚠️ Error processing user {user_id}: {e}")
            continue

    # Selected techniques
    return tecniques_comparisons["original -> IQR -> gaussian" if signal == "Gsr" else "original -> IQR -> butterworth"]

In [7]:
def generate_latex_table(data, phase, output_doc, signal_order=None, test_order=None, dec=3):
    if signal_order is None:
        signal_order = list(data.keys())

    if test_order is None:
        first_signal = signal_order[0]
        test_order = list(data[first_signal].keys())

    lines = []

    lines.append(r"\begin{table*}[h]")
    lines.append(r"\centering")
    lines.append(r"\renewcommand{\arraystretch}{1.3}")
    lines.append(r"\begin{tabular}{l l " + "c " * len(test_order) + "}")
    lines.append(r"\toprule")
    header = " & ".join([f"\\textbf{{{t}}}" for t in [
                        "Signal", "Comparison"] + test_order])
    lines.append(header + r" \\")
    lines.append(r"\midrule")

    for signal in signal_order:
        comparisons = list(data[signal][test_order[0]].keys())

        lines.append(
            f"\\multirow{{{len(comparisons)*2 - 1}}}{{*}}{{\\textbf{{{'EDA' if signal == 'Gsr' else 'HR'}}}}}"
        )
        for i, comp in enumerate(comparisons):
            parts = comp.replace(":", "").split("vs")
            comp_str = r"\makecell[l]{" + \
                " \\\\ / ".join([p.strip() for p in parts]) + "}"

            values = []
            for test in test_order:
                mean_r = data[signal][test][comp]["mean_r"]
                ns = data[signal][test][comp]["non_significant"]
                val_str = f"{mean_r:.{dec}f} / {ns}"
                values.append(val_str)

            line = "    & " + comp_str + " & " + " & ".join(values) + r" \\"
            lines.append(line)
            if i < len(comparisons) - 1:
                lines.append("    \\addlinespace")
        lines.append(r"\midrule")

    phase_label = "I" if phase == "first" else "II"
    extra_text = "Intra-device comparisons over the dashed line." if phase == "second" else ""

    lines.append(r"\end{tabular}")
    lines.append(
        fr"\caption{{Mean Pearson correlation between users for each Phase~{phase_label} "
        fr"event (number of non-significant participants on the right). {extra_text}}}"
    )
    lines.append(r"\label{tab:second_phase_correlations}" if phase == "second" else r"\label{tab:first_phase_correlations}")
    lines.append(r"\end{table*}")
    with open(output_doc, "w") as f:
        f.write("\n".join(lines))

## Generate pdfs and documents

In [8]:
import os

tasks = {
    "first": [
        ("full_test_start", "finish_baseline", "Baseline"),
        ("start_table", "finish_table", "SCT"),
        ("finish_table", "start_describe", "Relax"),
        ("start_describe", "finish_describe", "ST"),
        ("finish_describe", "full_test_end", "Cool down"),
        ("full_test_start", "full_test_end", "Full test"),
    ],
    "second": [
        ("relax_start", "relax_end", "Baseline"),
        ("squat_test_start", "squat_test_end", "Squat test"),
        ("sit_start", "sit_end", "Relax 1"),
        ("video1_start", "video1_end", "Video 1"),
        ("video1_end", "video2_start", "Relax 2"),
        ("video2_start", "video2_end", "Video 2"),
        ("full_test_start", "full_test_end", "Full test"),
    ]}

signals = {
    "first": ["Gsr", "Hr"],
    "second": ["Gsr", "Hr"]
}

device_names = {
    "first": ["emotibit", "shimmer", "empatica"],
    "second": ["emotibit_volar", "emotibit_dorsal", "shimmer_empatica", "shimmer_fingers"]
}

colors = {
    "first": ['dodgerblue', 'orange', 'limegreen', 'crimson', 'gold'],
    "second": ['gold', 'teal', 'crimson', 'purple', 'magenta', 'darkred']
}

comparisons = {
    "first": [("emotibit", "empatica"),
              ("empatica", "shimmer"),
              ("emotibit", "shimmer")],
    "second": [
        ("emotibit_volar", "emotibit_dorsal"),
        ("shimmer_empatica", "shimmer_fingers"),
        ("emotibit_volar", "shimmer_empatica"),
        ("emotibit_volar", "shimmer_fingers"),
        ("emotibit_dorsal", "shimmer_empatica"),
        ("emotibit_dorsal", "shimmer_fingers"),
    ]}

comparisons_ticks = {
    "first": {
        "Gsr": ["Emotibit vs Empatica", "Empatica vs Shimmer", "Emotibit vs Shimmer"],
        "Hr": ["Emotibit vs Empatica", "Empatica vs Shimmer", "Emotibit vs Shimmer"]},
    "second": {
        "Gsr": [
            "Emotibit:\nVolar vs\nDorsal",
            "Shimmer:\nVolar vs\nFingers",
            "Emotibit:\nVolar vs\nShimmer:\nVolar",
            "Emotibit:\nVolar vs\nShimmer:\nFingers",
            "Emotibit:\nDorsal vs\nShimmer:\nVolar",
            "Emotibit:\nDorsal vs\nShimmer:\nFingers",
        ],
        "Hr": [
            "Emotibit:\nVolar vs\nDorsal",
            "Shimmer:\nDorsal vs\nFingers",
            "Emotibit:\nVolar vs\nShimmer:\nDorsal",
            "Emotibit:\nVolar vs\nShimmer:\nFingers",
            "Emotibit:\nDorsal vs\nShimmer:\nDorsal",
            "Emotibit:\nDorsal vs\nShimmer:\nFingers",
        ]}}

comparisons_names_table = {
    "first":
    {
        "Gsr": {"empatica": "Empatica",
                "shimmer": "Shimmer",
                "emotibit": "Emotibit"},
        "Hr": {"empatica": "Empatica",
               "shimmer": "Shimmer",
               "emotibit": "Emotibit"}
    },
    "second":
    {
        "Gsr": {
            "emotibit_volar": "Emotibit: Volar",
            "emotibit_dorsal": "Emotibit: Dorsal",
            "shimmer_empatica": "Shimmer: Volar",
            "shimmer_fingers": "Shimmer: Fingers",
        },
        "Hr": {
            "emotibit_volar": "Emotibit: Volar",
            "emotibit_dorsal": "Emotibit: Dorsal",
            "shimmer_empatica": "Shimmer: Dorsal",
            "shimmer_fingers": "Shimmer: Fingers",
        }
    }
}

for phase in ["first","second"]:
    with open(f"processed_data/processed_data_{phase}.pickle", "rb") as file:
        data = pickle.load(file)

    table = {}
    table_dir = f'results/table/{phase}'
    os.makedirs(table_dir, exist_ok=True)
    for signal in signals[phase]:
        table[signal] = {}
        device_list = device_names[phase]
        comparison_list = comparisons[phase]
        comparison_tick = comparisons_ticks[phase][signal]
        comparisons_name = comparisons_names_table[phase][signal]
        color_list = colors[phase]

        scatter_dir = f'results/scatter/{phase}/{signal}'
        os.makedirs(scatter_dir, exist_ok=True)

        n_means_dir = f'results/n_means/{phase}'
        os.makedirs(n_means_dir, exist_ok=True)

        for task in tasks[phase]:
            plot_scatter_pearson(data,
                                 phase=phase,
                                 signal=signal,
                                 device_names=device_list,
                                 comparisons=comparison_list,
                                 comparisons_tick=comparison_tick,
                                 colors=color_list,
                                 output_pdf=f'{scatter_dir}/{task[2]}.pdf',
                                 task=task)

        plot_n_means(data,
                     phase=phase,
                     device_names=device_list,
                     # Exclude "Full test" from n_means plots
                     tasks=[t for t in tasks[phase] if t[2] != "Full test"],
                     signal=signal,
                     output_pdf=f'{n_means_dir}/{signal}_behaviour.pdf',
                     n_means=8)

        for task in tasks[phase]:
            table[signal][task[2]] = get_table_data(
                data, phase, signal=signal, device_names=device_list, comparisons=comparison_list, comparisons_name=comparisons_name, crop=(task[0], task[1]))

    generate_latex_table(table, phase,
                         output_doc=f'{table_dir}/correlation_table.txt')

✅ PDF generado: results/scatter/first/Gsr/Baseline.pdf


C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\3401110285.py:116: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\3401110285.py:116: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\3401110285.py:116: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\3401110285.py:116: NearConstantInputWarning: An input array is nearly constant; the computed correlation coefficient may be inaccurate.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\3401110285.py:116: ConstantInputWarning: An input array is constant; the correlation coeff

✅ PDF generado: results/scatter/first/Gsr/SCT.pdf


C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\3401110285.py:116: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\3401110285.py:116: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\3401110285.py:116: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\3401110285.py:116: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\3401110285.py:116: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(align

✅ PDF generado: results/scatter/first/Gsr/Relax.pdf


...

⚠️ Error en técnica 1 Hr original: list index out of range


C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\2141012916.py:95: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\2141012916.py:95: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\2141012916.py:95: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\2141012916.py:95: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\2141012916.py:95: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, 

⚠️ Error en técnica 1 Hr original: list index out of range
⚠️ Error en técnica 1 Hr original: list index out of range


C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\2141012916.py:95: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\2141012916.py:95: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\2141012916.py:95: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\2141012916.py:95: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, aligned2)
C:\Users\Usuario\AppData\Local\Temp\ipykernel_21180\2141012916.py:95: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(aligned1, 

⚠️ Error en técnica 1 Hr original: list index out of range


## Generate pdfs for Latex

### Scatter

In [ ]:
import os
from PyPDF2 import PdfReader, PdfWriter

pdf_files = {
    "gsr_first_phase": os.path.join("results/scatter", "first", "Gsr", "Full test.pdf"),
    "hr_first_phase": os.path.join("results/scatter", "first", "Hr", "Full test.pdf"),
    "gsr_second_phase": os.path.join("results/scatter", "second", "Gsr", "Full test.pdf"),
    "hr_second_phase": os.path.join("results/scatter", "second", "Hr", "Full test.pdf"),
}

output_path = os.path.join("results/scatter", "latex_pdfs")
os.makedirs(output_path, exist_ok=True)

for key, pdf_path in pdf_files.items():
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)

    # Third-to-last page
    page_index = total_pages - 4
    if page_index < 0:
        raise ValueError(f"The PDF {pdf_path} does not have enough pages.")

    writer = PdfWriter()
    writer.add_page(reader.pages[page_index])

    output_file = os.path.join(output_path, f"{key}.pdf")
    with open(output_file, "wb") as f_out:
        writer.write(f_out)

    print(f"{key} saved to {output_file}")

gsr_first_phase guardado en results/scatter\latex_pdfs\gsr_first_phase.pdf
hr_first_phase guardado en results/scatter\latex_pdfs\hr_first_phase.pdf
gsr_second_phase guardado en results/scatter\latex_pdfs\gsr_second_phase.pdf
hr_second_phase guardado en results/scatter\latex_pdfs\hr_second_phase.pdf


### Behavior

In [ ]:
import os
import fitz  # PyMuPDF

MM_TO_PT = 72.0 / 25.4  # ≈ 2.83465


def apply_trim_to_page(page, trim_mm):
    L_mm, B_mm, R_mm, T_mm = trim_mm
    L = float(L_mm) * MM_TO_PT
    B = float(B_mm) * MM_TO_PT
    R = float(R_mm) * MM_TO_PT
    T = float(T_mm) * MM_TO_PT

    rect = page.rect
    new_rect = fitz.Rect(rect)
    new_rect.x0 += L
    new_rect.y0 += B
    new_rect.x1 -= R
    new_rect.y1 -= T

    new_rect.x0 = max(new_rect.x0, rect.x0)
    new_rect.y0 = max(new_rect.y0, rect.y0)
    new_rect.x1 = min(new_rect.x1, rect.x1)
    new_rect.y1 = min(new_rect.y1, rect.y1)

    if new_rect.width <= 1 or new_rect.height <= 1:
        raise ValueError("Invalid trim. Check the measurements.")
    return new_rect


def crop_pdf(input_path, out_suffix_map, phase, out_dir):
    if not os.path.isfile(input_path):
        print(f"[WARNING] Does not exist: {input_path}")
        return

    doc = fitz.open(input_path)
    total_pages = len(doc)

    # Determine page index
    if total_pages == 1:
        page_index = 0
    elif total_pages == 2:
        page_index = 1
    else:
        page_index = total_pages - 2

    page = doc[page_index]
    base, _ = os.path.splitext(os.path.basename(input_path))
    base = base.lower()  # ensure lowercase

    os.makedirs(out_dir, exist_ok=True)

    for name, trim_mm in out_suffix_map.items():
        try:
            new_rect = apply_trim_to_page(page, trim_mm)
            out_doc = fitz.open()
            out_page = out_doc.new_page(
                width=new_rect.width, height=new_rect.height)
            out_page.show_pdf_page(
                out_page.rect, doc, page_index, clip=new_rect)

            # Build final name: <type>_behaviour_<phase>_<sensor>.pdf
            final_name = f"{base}_{phase}_{name}.pdf"
            out_path = os.path.join(out_dir, final_name)

            out_doc.save(out_path)
            out_doc.close()
            print(f"[OK] Saved: {out_path}")
        except Exception as e:
            print(f"[ERROR] {e}")

    doc.close()


# ---- Crop configurations ----
CROPS_PHASE1 = {
    "empatica": (1.0, 1, 158, 1.7),
    "shimmer": (80.0, 1, 79.0, 1.7),
    "emotibit": (159, 1, 0.0, 1.7),
}

CROPS_PHASE2 = {
    "emotibit_volar": (1.0, 1, 174, 1.7),
    "emotibit_dorsal": (58.9, 1, 116.1, 1.7),
    "shimmer_volar": (116.8, 1, 58.2, 1.7),
    "shimmer_finger": (174.7, 1, 0.3, 1.7),
}

# ---- Input paths ----
INPUTS = [
    ("./results/n_means/first/Gsr_behaviour.pdf",
     CROPS_PHASE1, 1, "./results/n_means/latex_pdfs/phase_1"),
    ("./results/n_means/first/Hr_behaviour.pdf",
     CROPS_PHASE1, 1, "./results/n_means/latex_pdfs/phase_1"),
    ("./results/n_means/second/Gsr_behaviour.pdf",
     CROPS_PHASE2, 2, "./results/n_means/latex_pdfs/phase_2"),
    ("./results/n_means/second/Hr_behaviour.pdf",
     CROPS_PHASE2, 2, "./results/n_means/latex_pdfs/phase_2"),
]

# ---- Create folders ----
os.makedirs("./results/n_means/latex_pdfs/phase_1", exist_ok=True)
os.makedirs("./results/n_means/latex_pdfs/phase_2", exist_ok=True)

# ---- Process PDFs ----
for input_path, crops, phase, out_dir in INPUTS:
    crop_pdf(input_path, crops, phase, out_dir)

[OK] Guardado: ./results/n_means/latex_pdfs/phase_1\gsr_behaviour_1_empatica.pdf
[OK] Guardado: ./results/n_means/latex_pdfs/phase_1\gsr_behaviour_1_shimmer.pdf
[OK] Guardado: ./results/n_means/latex_pdfs/phase_1\gsr_behaviour_1_emotibit.pdf
[OK] Guardado: ./results/n_means/latex_pdfs/phase_1\hr_behaviour_1_empatica.pdf
[OK] Guardado: ./results/n_means/latex_pdfs/phase_1\hr_behaviour_1_shimmer.pdf
[OK] Guardado: ./results/n_means/latex_pdfs/phase_1\hr_behaviour_1_emotibit.pdf
[OK] Guardado: ./results/n_means/latex_pdfs/phase_2\gsr_behaviour_2_emotibit_volar.pdf
[OK] Guardado: ./results/n_means/latex_pdfs/phase_2\gsr_behaviour_2_emotibit_dorsal.pdf
[OK] Guardado: ./results/n_means/latex_pdfs/phase_2\gsr_behaviour_2_shimmer_volar.pdf
[OK] Guardado: ./results/n_means/latex_pdfs/phase_2\gsr_behaviour_2_shimmer_finger.pdf
[OK] Guardado: ./results/n_means/latex_pdfs/phase_2\hr_behaviour_2_emotibit_volar.pdf
[OK] Guardado: ./results/n_means/latex_pdfs/phase_2\hr_behaviour_2_emotibit_dorsal.pd